# Диагностика фактической активности и закрытия гексагонов

## Назначение

Проверка эмпирического исчезновения потока заявок после изменения конфигурации гексагона и сопоставление с административным закрытием.

## Входные данные

`application_dataset.csv`, `hexagons_dataset.csv`; полная панель `hex × date` через `build_hex_activity_panel`.

## Результаты

Классификация эмпирической активности, событийного времени графики, CSV в `outputs/hex_activity/`.

## Статус

Диагностический ноутбук, входит в пайплайн проверки закрытий.


Административное закрытие задаётся через `region_id_new == -100`.
Строится **полная** панель `hex × date` (включая дни без заявок).

Важно:
- нулевые дни используются **только** для диагностики активности;
- они **не** трактуются как нулевые конверсии;
- эмпирический статус **не** заменяет административный `change_type` и не меняет воздействие assignment основной DiD.

Пороги — диагностические, а не универсальные бизнес-правила.


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from last_mile.filter import CLOSED, CORE_CHANGE_TYPES, STUDY_END, STUDY_START
from last_mile.io import load_applications, load_hexagons
from last_mile.hex_activity import (
    PRIMARY_WINDOW_DAYS,
    MIN_PRE_ORDERS,
    MIN_PRE_ACTIVE_DAYS,
    NEAR_CLOSED_RATIO,
    MAX_NEAR_CLOSED_ACTIVE_DAYS,
    EMPIRICAL_CLOSED,
    CONTINUES_ACTIVE,
    build_hex_activity_panel,
    build_event_time_activity,
    build_metadata_vs_empirical_crosstab,
    classify_empirical_activity,
    compute_closure_window_metrics,
    prepare_analysis_hex_universe,
    run_closure_threshold_sensitivity,
    save_activity_panel,
    selective_attrition_diagnostics,
    summarize_comparison_groups,
)
from last_mile.plot_style import PALETTE, save_figure, style_axes, with_plot_style

OUT_DIR = PROJECT_ROOT / "outputs"
FIG_DIR = PROJECT_ROOT / "figures" / "empirical"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Diagnostic thresholds (not universal business rules).
print({
    "PRIMARY_WINDOW_DAYS": PRIMARY_WINDOW_DAYS,
    "MIN_PRE_ORDERS": MIN_PRE_ORDERS,
    "MIN_PRE_ACTIVE_DAYS": MIN_PRE_ACTIVE_DAYS,
    "NEAR_CLOSED_RATIO": NEAR_CLOSED_RATIO,
    "MAX_NEAR_CLOSED_ACTIVE_DAYS": MAX_NEAR_CLOSED_ACTIVE_DAYS,
})
print("OUT_DIR:", OUT_DIR)
print("FIG_DIR:", FIG_DIR)

{'PRIMARY_WINDOW_DAYS': 28, 'MIN_PRE_ORDERS': 5, 'MIN_PRE_ACTIVE_DAYS': 3, 'NEAR_CLOSED_RATIO': 0.05, 'MAX_NEAR_CLOSED_ACTIVE_DAYS': 1}
OUT_DIR: <under repository root>
FIG_DIR: <under repository root>


## 1. Загрузка через единый loader и полный `hex_activity_panel`

In [2]:
applications = load_applications()
hexagons = load_hexagons()

prepared = prepare_analysis_hex_universe(applications, hexagons)
hex_meta = prepared["hex_meta"]
apps_clean = prepared["apps_clean"]

print("hexes:", hex_meta["hex"].nunique())
print("apps in analysis window:", len(apps_clean))
print("treated hexes:", int(hex_meta["is_treated"].sum()))
print("change_type:\n", hex_meta["change_type"].value_counts())
assert hex_meta["hex"].is_unique

[INFO] attach_hex_metadata: отброшено 109 заявок из гексагонов, не представленных в hexagons_dataset.
[INFO] exclude_cohorts: исключено 683082 заявок из когорт ['2022-10-19'].


hexes: 13255
apps in analysis window: 1104494
treated hexes: 9327
change_type:
 change_type
other                  5039
region_and_workmode    3154
workmode_only          2682
region_only            1288
closed                  488
opened                  383
never_active            221
Name: count, dtype: int64


In [3]:
hex_activity_panel = build_hex_activity_panel(
    hex_meta=hex_meta,
    apps_clean=apps_clean,
    study_start=STUDY_START,
    study_end=STUDY_END,
)

print("rows:", f"{len(hex_activity_panel):,}")
print("zero days:", f"{(hex_activity_panel['n_orders'] == 0).sum():,}")
print("sum n_orders:", int(hex_activity_panel["n_orders"].sum()))
print("unique hex×date:", hex_activity_panel.duplicated(["hex", "date"]).sum() == 0)

panel_paths = save_activity_panel(hex_activity_panel, OUT_DIR)
panel_paths

rows: 2,664,255
zero days: 2,361,857
sum n_orders: 1104494


unique hex×date: True


{'summary': WindowsPath('C:<PROJECT_ROOT>/…'),
 'panel': WindowsPath('C:<PROJECT_ROOT>/…')}

## 2–3. Метрики окон и эмпирическая классификация

In [4]:
metrics_by_window = pd.concat(
    [
        compute_closure_window_metrics(hex_activity_panel, window_days=w)
        for w in (14, 28, 56)
    ],
    ignore_index=True,
)
metrics_by_window.to_csv(OUT_DIR / "hex_closure_diagnostics_by_window.csv", index=False)
metrics_by_window.head()

               hex  window_days treatment_date   change_type  \
0  880b3086a9fffff           14     2022-08-12        opened   
1  880b3086abfffff           14     2022-08-12        opened   
2  880b3086e7fffff           14     2022-08-12        opened   
3  880b30adadfffff           14     2022-08-12  never_active   
4  880b340491fffff           14     2022-08-12         other   

             subject     cohort  pre_observed_days  post_observed_days  \
0  Тюменская область 2022-08-12                 14                  14   
1  Тюменская область 2022-08-12                 14                  14   
2  Тюменская область 2022-08-12                 14                  14   
3  Тюменская область 2022-08-12                 14                  14   
4  Тюменская область 2022-08-12                 14                  14   

   pre_orders  post_orders  ...  pre_zero_share  post_zero_share  \
0          16            3  ...        0.428571         0.857143   
1           0            4  ...   

In [5]:
classification = classify_empirical_activity(
    metrics_by_window, window_days=PRIMARY_WINDOW_DAYS
)
classification.to_csv(OUT_DIR / "hex_empirical_closure_classification.csv", index=False)
classification["empirical_status"].value_counts()

empirical_status
insufficient_post_window    4809
sparse_pre_activity         3231
continues_active            1221
empirical_closed              65
near_closed                    1
Name: count, dtype: int64

## 4. Чувствительность порогов

In [6]:
sensitivity = run_closure_threshold_sensitivity(metrics_by_window)
sensitivity.to_csv(OUT_DIR / "closure_threshold_sensitivity.csv", index=False)
agreement = sensitivity.attrs.get("agreement")
if agreement is not None and not agreement.empty:
    agreement.to_csv(OUT_DIR / "closure_threshold_sensitivity_agreement.csv", index=False)
    display(agreement.head(20))
sensitivity.sort_values(["window_days", "min_pre_orders", "near_closed_ratio"])

    compare_window_days  compare_min_pre_orders  compare_near_closed_ratio  \
0                    14                       3                       0.00   
1                    14                       3                       0.05   
2                    14                       3                       0.10   
3                    14                       5                       0.00   
4                    14                       5                       0.05   
5                    14                       5                       0.10   
6                    14                      10                       0.00   
7                    14                      10                       0.05   
8                    14                      10                       0.10   
9                    28                       3                       0.00   
10                   28                       3                       0.05   
11                   28                       3                 

    window_days  min_pre_orders  near_closed_ratio  n_hexagons  \
0            14               3               0.00        9327   
1            14               3               0.05        9327   
2            14               3               0.10        9327   
3            14               5               0.00        9327   
4            14               5               0.05        9327   
5            14               5               0.10        9327   
6            14              10               0.00        9327   
7            14              10               0.05        9327   
8            14              10               0.10        9327   
9            28               3               0.00        9327   
10           28               3               0.05        9327   
11           28               3               0.10        9327   
12           28               5               0.00        9327   
13           28               5               0.05        9327   
14        

## 5. Сопоставление с административной классификацией

In [7]:
crosstab = build_metadata_vs_empirical_crosstab(classification)
crosstab.to_csv(OUT_DIR / "metadata_vs_empirical_closure.csv", index=False)
display(crosstab.pivot(index="change_type", columns="empirical_status", values="n_hexagons").fillna(0).astype(int))

groups = summarize_comparison_groups(classification, hex_activity_panel)
groups["group_summary"].to_csv(OUT_DIR / "metadata_vs_empirical_group_summary.csv", index=False)
groups["core_empirically_closed"].to_csv(OUT_DIR / "core_empirically_closed_hexagons.csv", index=False)
groups["administratively_closed_but_active"].to_csv(
    OUT_DIR / "administratively_closed_but_active_hexagons.csv", index=False
)
groups["group_summary"]

empirical_status     continues_active  empirical_closed  \
change_type                                               
closed                              7                 0   
never_active                       11                 1   
opened                             22                 4   
other                             763                29   
region_and_workmode                74                12   
region_only                        35                 6   
workmode_only                     309                13   

empirical_status     insufficient_post_window  near_closed  \
change_type                                                  
closed                                    279            0   
never_active                               19            0   
opened                                     23            0   
other                                    1895            1   
region_and_workmode                      1502            0   
region_only                       

                               group  n_hexagons  pre_orders  post_orders  \
0  admin_closed_and_empirical_closed           0           0            0   
1   admin_closed_but_orders_continue         186        4877         1105   
2          core_and_empirical_closed          31         693          129   
3          core_and_continues_active         418       60039        27189   

                                 cohort_distribution  \
0                                                      
1  {'2022-08-12': 21, '2022-08-19': 33, '2022-09-...   
2  {'2022-07-27': 2, '2022-08-12': 13, '2022-08-1...   
3  {'2022-07-27': 41, '2022-08-12': 69, '2022-08-...   

                            change_type_distribution  \
0                                                      
1                                    {'closed': 186}   
2  {'region_and_workmode': 12, 'region_only': 6, ...   
3  {'region_and_workmode': 74, 'region_only': 35,...   

                                       example_hexes

## 6–7. Событийное время диагностика и графики

In [8]:
event_time = build_event_time_activity(
    hex_activity_panel, classification, bootstrap_reps=200
)
event_time.to_csv(OUT_DIR / "hex_activity_event_time.csv", index=False)

attrition = selective_attrition_diagnostics(hex_activity_panel)
attrition.to_csv(OUT_DIR / "hex_selective_attrition_diagnostics.csv", index=False)

GROUP_LABELS = {
    "admin_closed": "Административно closed",
    "core": "CORE",
    "empirical_closed": "Эмпирически закрытые",
    "continues_active": "Активность продолжается",
}
GROUP_COLORS = {
    "admin_closed": "#7B8794",
    "core": PALETTE["region_and_workmode"],
    "empirical_closed": "#B85C38",
    "continues_active": PALETTE["region_only"],
}

@with_plot_style
def plot_event_orders(df):
    fig, ax = plt.subplots(figsize=(8.2, 4.4))
    for gname, g in df.groupby("group"):
        ax.plot(g["relative_day"], g["mean_n_orders"], label=GROUP_LABELS[gname],
                color=GROUP_COLORS[gname], linewidth=1.6)
    ax.axvline(0, color=PALETTE["zero"], linewidth=0.9, linestyle=":")
    ax.set_xlabel("День относительно даты изменения")
    ax.set_ylabel("Среднее число заявок")
    ax.set_title("Динамика заявок: событийное время")
    ax.legend(loc="upper right", frameon=False)
    style_axes(ax)
    save_figure(fig, FIG_DIR / "hex_activity_event_time_orders.pdf", preview_dpi=300)

@with_plot_style
def plot_event_active(df):
    fig, ax = plt.subplots(figsize=(8.2, 4.4))
    for gname, g in df.groupby("group"):
        ax.plot(g["relative_day"], g["active_share"], label=GROUP_LABELS[gname],
                color=GROUP_COLORS[gname], linewidth=1.6)
        ax.fill_between(g["relative_day"], g["active_share_ci_low"], g["active_share_ci_high"],
                        color=GROUP_COLORS[gname], alpha=0.12, linewidth=0)
    ax.axvline(0, color=PALETTE["zero"], linewidth=0.9, linestyle=":")
    ax.set_xlabel("День относительно даты изменения")
    ax.set_ylabel("Доля активных гексагонов")
    ax.set_title("Доля дней с заявками: событийное время")
    ax.legend(loc="upper right", frameon=False)
    style_axes(ax)
    save_figure(fig, FIG_DIR / "hex_activity_event_time_active_share.pdf", preview_dpi=300)

@with_plot_style
def plot_pre_post(cls):
    fig, ax = plt.subplots(figsize=(6.8, 5.4))
    type_colors = {
        "closed": "#7B8794",
        "region_and_workmode": PALETTE["region_and_workmode"],
        "region_only": PALETTE["region_only"],
        "workmode_only": PALETTE["workmode_only"],
        "opened": "#A0AEC0",
        "other": "#CBD5E0",
    }
    for ctype, g in cls.groupby("change_type"):
        ax.scatter(np.log1p(g["pre_orders"]), np.log1p(g["post_orders"]),
                   s=12, alpha=0.45, label=ctype,
                   color=type_colors.get(ctype, PALETTE["text_muted"]), edgecolors="none")
    lim = max(np.log1p(cls["pre_orders"]).max(), np.log1p(cls["post_orders"]).max())
    ax.plot([0, lim], [0, lim], color=PALETTE["zero"], linewidth=0.9, linestyle="--")
    ax.set_xlabel("log(1 + заявки до изменения)")
    ax.set_ylabel("log(1 + заявки после изменения)")
    ax.set_title("Заявки до и после изменения (окно 28 дней)")
    ax.legend(loc="upper left", fontsize=7, frameon=False, ncol=2)
    style_axes(ax)
    save_figure(fig, FIG_DIR / "hex_pre_vs_post_orders.pdf", preview_dpi=300)

@with_plot_style
def plot_last_request(cls):
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.hist(cls["last_request_relative_day"].dropna(), bins=40,
            color=PALETTE["treated"], alpha=0.85, edgecolor="white")
    ax.axvline(0, color=PALETTE["zero"], linewidth=0.9, linestyle=":")
    ax.set_xlabel("Относительный день последней заявки")
    ax.set_ylabel("Число гексагонов")
    ax.set_title("Распределение дня последней заявки")
    style_axes(ax)
    save_figure(fig, FIG_DIR / "hex_last_request_relative_distribution.pdf", preview_dpi=300)

@with_plot_style
def plot_heatmap(cls):
    pivot = cls.groupby(["change_type", "empirical_status"]).size().unstack(fill_value=0)
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    sns.heatmap(pivot, annot=True, fmt="d", cmap="Blues", ax=ax,
                cbar_kws={"label": "Число гексагонов"})
    ax.set_xlabel("Эмпирический статус")
    ax.set_ylabel("Административный change_type")
    ax.set_title("Сопоставление административного и эмпирического статуса")
    save_figure(fig, FIG_DIR / "metadata_vs_empirical_closure_heatmap.pdf", preview_dpi=300)

@with_plot_style
def plot_sensitivity(sens):
    fig, ax = plt.subplots(figsize=(7.5, 4.4))
    for ratio, g in sens.groupby("near_closed_ratio"):
        for min_pre, gg in g.groupby("min_pre_orders"):
            ax.plot(gg["window_days"], gg["n_empirical_closed"], marker="o", linewidth=1.4,
                    label=f"pre≥{min_pre}, ratio={ratio:g}")
    ax.set_xlabel("Окно, дней")
    ax.set_ylabel("Число empirical_closed")
    ax.set_title("Чувствительность классификации закрытия")
    ax.legend(loc="best", fontsize=7, frameon=False, ncol=2)
    style_axes(ax)
    save_figure(fig, FIG_DIR / "closure_threshold_sensitivity.pdf", preview_dpi=300)

plot_event_orders(event_time)
plot_event_active(event_time)
plot_pre_post(classification)
plot_last_request(classification)
plot_heatmap(classification)
plot_sensitivity(sensitivity)

n_admin_closed = int((classification["change_type"] == CLOSED).sum())
n_admin_emp = int(((classification["change_type"] == CLOSED) & (classification["empirical_status"] == EMPIRICAL_CLOSED)).sum())
n_admin_active = int(((classification["change_type"] == CLOSED) & (classification["post_orders"] > 0)).sum())
n_core_emp = int((classification["change_type"].isin(CORE_CHANGE_TYPES) & (classification["empirical_status"] == EMPIRICAL_CLOSED)).sum())
headline = pd.DataFrame([{
    "n_admin_closed": n_admin_closed,
    "n_admin_closed_empirical_closed": n_admin_emp,
    "n_admin_closed_orders_continue": n_admin_active,
    "n_core_empirical_closed": n_core_emp,
    "n_treated_classified": int(classification["hex"].nunique()),
}])
headline.to_csv(OUT_DIR / "hex_closure_headline_counts.csv", index=False)
headline

,n_admin_closed,n_admin_closed_empirical_closed,n_admin_closed_orders_continue,n_core_empirical_closed,n_treated_classified
0,374,0,186,31,9327


## 10. Риск селективного исчезновения

Для контрольной группы используется cohort-aligned сравнение по датам когорт воздействия (без случайных псевдодат).

In [9]:
attrition_summary = (
    attrition.groupby(["sample", "change_type"], dropna=False)
    .agg(
        n_hexagons=("n_hexagons", "sum"),
        n_zero_post_orders=("n_zero_post_orders", "sum"),
        mean_share_zero=("share_zero_post_orders", "mean"),
    )
    .reset_index()
)
attrition_summary

                   sample          change_type  n_hexagons  \
0  control_cohort_aligned        never_treated       15712   
1                 treated               closed         374   
2                 treated         never_active         210   
3                 treated               opened         347   
4                 treated                other        3886   
5                 treated  region_and_workmode        2232   
6                 treated          region_only         530   
7                 treated        workmode_only        1748   

   n_zero_post_orders  mean_share_zero  
0                6115         0.389193  
1                 188         0.594956  
2                 111         0.491889  
3                 183         0.522144  
4                 999         0.240439  
5                 967         0.405016  
6                 159         0.209928  
7                 618         0.361198  

## Вывод

Эмпирическое исчезновение заявок после даты изменения согласуется с административным закрытием не для всех гексагонов: часть CORE-гексагонов сохраняет активность в post-окне. Диагностика используется для оценки риска селективного attrition и чувствительности основной спецификации, но не переопределяет воздействие assignment.

Для контрольной группы применяется cohort-aligned сравнение по датам когорт воздействия (без случайных псевдодат).
